# 🔬 Pattern Exploration Lab

This notebook provides advanced tools for exploring fake news detection patterns in depth. Use this for:
- Analyzing your own article datasets
- Experimenting with different detection algorithms
- Creating custom visualizations
- Building new pattern detection features

In [ ]:
# Setup and imports
import sys
sys.path.append('..')

from src.data_loader import NewsDataLoader
from src.pattern_analyzer import PatternAnalyzer
from src.visualizer import PatternVisualizer

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import json

%matplotlib inline

# Initialize tools
loader = NewsDataLoader()
analyzer = PatternAnalyzer()
visualizer = PatternVisualizer()

print("🧪 Pattern exploration tools loaded!")


## 📊 Bulk Analysis

Analyze all sample articles and compare patterns:

In [ ]:
# Load and analyze all articles
articles_df = loader.load_sample_articles()

# Perform analysis on all articles
all_analyses = []
enhanced_articles = []

for idx, article in articles_df.iterrows():
    analysis = analyzer.analyze_article(
        article['title'],
        article['content'],
        article['source']
    )

    all_analyses.append(analysis)

    # Enhanced article data for visualization
    enhanced_article = dict(article)
    enhanced_article['credibility_score'] = analysis['overall_score']
    enhanced_article['title_sensationalism'] = analysis['title_analysis']['sensationalism_score']
    enhanced_article['credibility_markers'] = analysis['content_analysis'].get('credibility_markers', 0)
    enhanced_article['vague_sources'] = analysis['content_analysis'].get('vague_sources', 0)

    enhanced_articles.append(enhanced_article)

enhanced_df = pd.DataFrame(enhanced_articles)
print(f"✅ Analyzed {len(enhanced_articles)} articles")


## 📈 Comparative Visualizations

In [ ]:
# Plot credibility scores comparison
fig = visualizer.plot_credibility_scores(enhanced_articles)
plt.show()

print("💡 Notice how fabricated articles tend to have lower credibility scores")


In [ ]:
# Create comprehensive pattern summary
fig = visualizer.create_pattern_summary_chart(all_analyses)
plt.show()

print("🔍 Compare patterns across all articles side-by-side")


## 📋 Statistical Analysis

In [ ]:
# Statistical comparison between real and fake articles
real_articles = enhanced_df[enhanced_df['is_fabricated'] == False]
fake_articles = enhanced_df[enhanced_df['is_fabricated'] == True]

print("📊 STATISTICAL COMPARISON:")
print("\n🟢 Real Articles:")
print(f"  Average credibility score: {real_articles['credibility_score'].mean():.3f}")
print(f"  Average title sensationalism: {real_articles['title_sensationalism'].mean():.3f}")
print(f"  Average credibility markers: {real_articles['credibility_markers'].mean():.1f}")
print(f"  Average vague sources: {real_articles['vague_sources'].mean():.1f}")

print("\n🔴 Fabricated Articles:")
print(f"  Average credibility score: {fake_articles['credibility_score'].mean():.3f}")
print(f"  Average title sensationalism: {fake_articles['title_sensationalism'].mean():.3f}")
print(f"  Average credibility markers: {fake_articles['credibility_markers'].mean():.1f}")
print(f"  Average vague sources: {fake_articles['vague_sources'].mean():.1f}")

print("\n💡 Key Differences:")
score_diff = real_articles['credibility_score'].mean() - fake_articles['credibility_score'].mean()
sensational_diff = fake_articles['title_sensationalism'].mean() - real_articles['title_sensationalism'].mean()
marker_diff = real_articles['credibility_markers'].mean() - fake_articles['credibility_markers'].mean()

print(f"  Real articles score {score_diff:.3f} points higher on credibility")
print(f"  Fake articles are {sensational_diff:.3f} points more sensational")
print(f"  Real articles have {marker_diff:.1f} more credibility markers on average")


## 🔤 Word Pattern Analysis

In [ ]:
# Analyze word patterns in titles
real_titles = ' '.join(real_articles['title'].tolist())
fake_titles = ' '.join(fake_articles['title'].tolist())

# Create word clouds
print("☁️ Word clouds showing common terms:")
print("\n🟢 Real news titles:")
fig1 = visualizer.create_word_cloud(real_titles, "Real News - Title Words")
if fig1:
    plt.show()

print("\n🔴 Fabricated news titles:")
fig2 = visualizer.create_word_cloud(fake_titles, "Fabricated News - Title Words")
if fig2:
    plt.show()


## 🛠️ Custom Analysis Tools

Add your own articles for analysis:

In [ ]:
# Add your own article for analysis
custom_article = {
    "title": "Replace this with your article title",
    "content": "Replace this with the article content you want to analyze...",
    "source": "Replace with source name"
}

# Analyze your custom article
def analyze_custom_article(article_dict):
    print(f"📄 Analyzing: {article_dict['title']}")

    analysis = analyzer.analyze_article(
        article_dict['title'],
        article_dict['content'],
        article_dict['source']
    )

    print(f"\n📊 Results:")
    print(f"  Credibility Score: {analysis['overall_score']:.3f}")
    print(f"  Title Sensationalism: {analysis['title_analysis']['sensationalism_score']:.3f}")
    print(f"  Suspicious Phrases: {analysis['title_analysis']['suspicious_phrases']}")

    if 'credibility_markers' in analysis['content_analysis']:
        print(f"  Credibility Markers: {analysis['content_analysis']['credibility_markers']}")
        print(f"  Vague Sources: {analysis['content_analysis']['vague_sources']}")

    # Create visualization
    fig = visualizer.plot_pattern_comparison(analysis)
    plt.show()

    return analysis

# Uncomment and modify the custom_article dict above, then run:
# custom_analysis = analyze_custom_article(custom_article)

print("💡 Modify the custom_article dictionary above and uncomment the last line to analyze your own content!")


## 🎯 Algorithm Tuning

Experiment with detection algorithm parameters:

In [ ]:
# Test different suspicious phrases
def test_suspicious_phrases(new_phrases):
    """Test how adding new suspicious phrases affects detection."""

    # Create modified analyzer
    modified_analyzer = PatternAnalyzer()
    modified_analyzer.suspicious_phrases.extend(new_phrases)

    print(f"🧪 Testing with additional phrases: {new_phrases}")

    # Re-analyze articles
    for idx, article in articles_df.iterrows():
        original_analysis = analyzer.analyze_article(
            article['title'], article['content'], article['source']
        )

        modified_analysis = modified_analyzer.analyze_article(
            article['title'], article['content'], article['source']
        )

        score_change = modified_analysis['overall_score'] - original_analysis['overall_score']

        if abs(score_change) > 0.01:  # Only show significant changes
            status = "🔴 FAKE" if article['is_fabricated'] else "🟢 REAL"
            print(f"{status} '{article['title'][:50]}...': {score_change:+.3f}")

# Example: Test new phrases
test_phrases = ["breakthrough", "secret", "revealed"]
test_suspicious_phrases(test_phrases)


## 💾 Export Results

Save your analysis results for further use:

In [ ]:
# Export enhanced dataset with analysis results
export_data = enhanced_df.to_dict('records')

# Save to JSON
with open('../data/analyzed_articles.json', 'w') as f:
    json.dump(export_data, f, indent=2)

# Also save as CSV for spreadsheet analysis
enhanced_df.to_csv('../data/analyzed_articles.csv', index=False)

print("💾 Analysis results exported to:")
print("  - ../data/analyzed_articles.json")
print("  - ../data/analyzed_articles.csv")

# Show summary
print(f"\n📈 Summary:")
print(f"  Total articles: {len(enhanced_df)}")
print(f"  Real articles: {len(real_articles)}")
print(f"  Fake articles: {len(fake_articles)}")
print(f"  Average detection accuracy: {((enhanced_df['is_fabricated'] == (enhanced_df['credibility_score'] < 0.5)).sum() / len(enhanced_df) * 100):.1f}%")


## 🚀 Next Experiments

Ideas for extending this analysis:

1. **Add More Features**: Analyze publication dates, author patterns, social media metrics
2. **Machine Learning**: Train classifiers on these features using scikit-learn
3. **Real-time Analysis**: Build a web interface for live article checking
4. **Dataset Expansion**: Add more diverse articles from different domains
5. **Cross-validation**: Test the algorithm on external datasets

Remember: The goal is to **augment human critical thinking**, not replace it! 🧠✨